# 行動解析 はじめて 🐞🐟

動物の動画から「どこを・どれくらいの速さで・どう曲がりながら」動いたかを数値にし、グラフにするノートブックです。
**上のセルから順に ▶（実行）を押していくだけ**で、最後に指標の表（CSV）と図（PNG）が手に入ります。

```
動画 ──(SLEAP / TrackMate / 手で記録)──▶ 座標のファイル
   ──▶ ① 読み込み ② fps とスケールの設定 ③ 前処理（外れ値・欠損・ぶれ）
   ──▶ ④ 指標の計算 ⑤ 図 ⑥ 保存
```

**最初は実データがなくても大丈夫。** 練習用の合成データ（3 匹: ランダムに歩く個体・壁ぞいに歩く個体・左右交互に曲がる個体）で全体を試せます。

> 💡 セルの中で `# ★` の印がある行は、自分の実験に合わせて書きかえる場所です。

## 0. 準備：解析プログラム（kodo）を読み込む

**何をする？** このノートブックで使う解析プログラム `kodo`（行動）を Colab に取り込みます。

**なぜ必要？** 速度や転向角の計算はたくさんの式が必要です。ここで一度読み込めば、あとは `kodo.〇〇()` と書くだけで使えます。

- GitHub から自動でダウンロードします（30 秒ほど）。
- うまくいかないとき（リポジトリが非公開など）は、先生から `behavior_analysis` フォルダの zip をもらい、左の 📁 にアップロードしてから `ZIP_FILE` にファイル名を入れて実行してください。

In [ ]:
import os, sys, subprocess, zipfile
from pathlib import Path

REPO_URL = "https://github.com/Punonmamoru/biology.git"
BRANCH = "main"          # ★ 先生に指示されたらブランチ名を変える
ZIP_FILE = None          # ★ zip をアップロードしたときは "behavior_analysis.zip" のように書く


def find_kodo():
    here = Path.cwd()
    for c in [here, here.parent, here / "behavior_analysis", here / "biology" / "behavior_analysis",
              Path("/content/biology/behavior_analysis")]:
        if (c / "kodo" / "__init__.py").exists():
            return c
    return None


try:
    import kodo
except ImportError:
    root = find_kodo()
    if root is None and ZIP_FILE:
        zipfile.ZipFile(ZIP_FILE).extractall(".")
        root = find_kodo()
    if root is None:
        subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", BRANCH, REPO_URL, "biology"], check=True)
        root = find_kodo()
    sys.path.insert(0, str(root))
    import kodo

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

print("kodo", kodo.__version__, "を読み込みました")

## 1. グラフの日本語を化けないようにする

**何をする？** 日本語のフォントを探して、グラフで使えるようにします。

**なぜ必要？** グラフを描くソフト（matplotlib）は、そのままだと日本語が「□□□」に化けます。Colab では最初の 1 回だけフォントのインストールに 1 分ほどかかります。

In [ ]:
FONT = kodo.setup_japanese_font()

## 2. データを用意する

**何をする？** 解析する座標のファイルを用意します。

- `USE_DEMO = True` … 練習用の合成データを作って使います（最初はこちら）。
- `USE_DEMO = False` … 自分のファイルをアップロードします。実行するとファイル選択の画面が出ます。

| 使ったソフト | アップロードするファイル |
|---|---|
| SLEAP | `File → Export Analysis HDF5` で作った `○○.analysis.h5` |
| Fiji / TrackMate | Spots の表を `Export to CSV` した `.csv`（`POSITION_X` などの列があるもの） |
| 手で記録 | 1 行目が `frame,x,y`（複数個体なら `id,frame,x,y`）の `.csv` |

あわせて、**定規を写した画像**（動物を撮ったのと同じカメラ位置・同じ高さで撮ったもの）があればアップロードします。mm に換算するのに使います。

**なぜ必要？** 形式が違っても、`kodo` は全部「個体 × フレーム × 部位 × (x, y)」という同じ形にそろえるので、このあとの手順は共通です。

In [ ]:
USE_DEMO = True          # ★ 自分のデータを使うときは False

if USE_DEMO:
    demo_paths = kodo.synthetic.export_demo_files("demo_data")
    DATA_FILE = demo_paths["sleap"]       # ★ "trackmate" や "manual" に変えると他の形式も試せる
    RULER_IMAGE = demo_paths["image"]
else:
    from google.colab import files
    print("座標のファイル（.h5 か .csv）を選んでください")
    DATA_FILE = next(iter(files.upload()))
    print("定規を写した画像を選んでください（なければキャンセル）")
    up = files.upload()
    RULER_IMAGE = next(iter(up)) if up else None

print("データ:", DATA_FILE)
print("定規の画像:", RULER_IMAGE)

## 3. フレームレート（fps）を設定して読み込む

**何をする？** 動画の **fps（1 秒あたりのコマ数）** を入れて、ファイルを読み込みます。

**なぜ必要？** 座標のファイルには「何フレーム目か」しか書かれていません。秒に直すには fps が必要です。
fps を間違えると、速度も時間もすべて同じ倍率で狂います（例: 本当は 60 fps なのに 30 と入れると、速度が半分になる）。

- スマホのスロー撮影（120 fps や 240 fps）は、再生用に 30 fps と記録されていることがあります。**撮影時の設定**を確かめましょう。
- 練習用データは 30 fps です。
- `Y_DOWN`：SLEAP・TrackMate・Fiji の座標は y 軸が下向きです（そのまま True）。y 軸が上向きのデータを True のまま読むと、転向角の左右が逆になります。

In [ ]:
FPS = 30                 # ★ 撮影した動画の fps
Y_DOWN = True            # ★ y 軸が下向き（画像の座標）なら True。ImageJ で "Invert Y coordinates" にしていたら False

tracks = kodo.load(DATA_FILE, fps=FPS, y_down=Y_DOWN)
print(tracks)

### 解析に使う部位を選ぶ

SLEAP では体の複数の点（頭・胸・腹など）を追跡します。速度や軌跡には **1 点だけ**を使います。
よく検出される、体の中心に近い部位（例: `thorax` 胸部）がおすすめです。`None` にすると、見えている部位の平均（重心）を使います。
TrackMate や手動追跡のデータは点が 1 つ（`point`）なので、`None` のままで構いません。

In [ ]:
NODE = "thorax" if "thorax" in tracks.node_names else None   # ★ 使う部位の名前
print("使える部位:", tracks.node_names, "→ 使う部位:", NODE)

## 4. まずは生のデータを見る

**何をする？** 読み込んだ座標をそのまま描き、よくある問題がないかを自動でチェックします。

**なぜ必要？** 計算の前に「目で見る」のがいちばん大事です。軌跡が急に飛んでいる（誤検出）、途中で途切れている（見失い）、2 匹の線が入れかわっている（ID の入れ替わり）などは、ここで気づけます。

In [ ]:
kodo.check_tracks(tracks, node=NODE)
fig, ax = kodo.plot_trajectories(tracks, node=NODE, title="生のデータ（前処理の前）")
plt.show()

## 5. スケールを設定する（px → mm）

**何をする？** 定規の画像で、**離れた 2 つの目盛りの画素の位置**を読み取り、1 px が何 mm かを計算します。

$$\text{mm/px} = \frac{\text{2 つの目盛りの間の長さ [mm]}}{\sqrt{(x_2-x_1)^2 + (y_2-y_1)^2}\ \text{[px]}}$$

**なぜ必要？** px のままだと、カメラの高さやズームが変わると数値が変わってしまい、他の実験や他の班と比べられません。

1. 下のセルを実行すると、目盛り（青い格子）つきで画像が出ます。`CROP` で一部を拡大できます。
2. 2 つの目盛り（例: 0 mm と 100 mm）の位置 (x, y) を読み取り、`RULER_P1`, `RULER_P2` に書きます。
3. もう一度実行し、赤い × が目盛りにぴったり重なっているか確かめます。

> 2 点はできるだけ離しましょう。読み取りの誤差 ±1 px の影響が小さくなります。練習用データの正解は (40, 500) と (540, 500)、長さ 100 mm です。

In [ ]:
RULER_P1 = (40, 500)     # ★ 1 つめの目盛りの (x, y) [px]
RULER_P2 = (540, 500)    # ★ 2 つめの目盛りの (x, y) [px]
RULER_LENGTH_MM = 100    # ★ 2 つの目盛りの間の長さ [mm]
CROP = None              # ★ 拡大するなら (x0, y0, x1, y1)。例: (0, 450, 680, 560)

if RULER_IMAGE is not None:
    fig, ax = kodo.show_image(RULER_IMAGE, grid_step=50, crop=CROP, points=[RULER_P1, RULER_P2])
    plt.show()
    MM_PER_PX = kodo.mm_per_px_from_points(RULER_P1, RULER_P2, RULER_LENGTH_MM)
    print(f"1 px = {MM_PER_PX:.4f} mm（1 mm = {1 / MM_PER_PX:.2f} px）")
else:
    MM_PER_PX = None
    print("⚠ 定規の画像がないので px のまま解析します。班どうし・実験どうしの比較には使えません。")

## 6. 前処理：外れ値 → 欠損の補間 → 平滑化

**何をする？** 追跡の失敗を取り除き、短い見失いを埋め、細かいぶれをならします。

**なぜ必要？** 追跡ソフトは完ぺきではありません。

| 問題 | 放っておくと | ここでの対策 |
|---|---|---|
| 誤検出（一瞬だけ別の場所に飛ぶ） | 速度・移動距離が大きくなりすぎる | 前後 3 フレームずつ（計 7 フレーム）の中央値の位置から `OUTLIER_PX` 以上離れた点を欠損にする（速く曲がった本物の点は消さないよう、許す距離はその付近の 1 フレームの移動距離の 3 倍以上にしてある） |
| 見失い（数フレーム座標がない） | その間の移動が数えられない | `MAX_GAP` フレーム以下の欠損だけを直線で埋める（長い欠損は埋めない＝見ていない動きを作らない） |
| 細かいぶれ（静止していても数 px 揺れる） | 静止中なのに「動いている」ことになる | Savitzky-Golay フィルタでならす（曲がり角がなまりにくい） |

**この順番にする理由：** 外れ値を残したまま補間・平滑化すると、その誤差が前後のフレームに広がってしまうからです。
数値は px 単位です（スケール換算の前に行うため）。目安: `OUTLIER_PX` は体長の半分くらい。

> ⚠ 総移動距離・平均の速さ・直進性・移動の割合は、fps と平滑化の強さで変わります（ぶれが毎フレーム足し込まれるため）。**比べるデータは、同じ fps・同じ設定で処理しましょう。**

In [ ]:
OUTLIER_PX = 15          # ★ これ以上離れた点を外れ値とする [px]（体長の半分くらい）
MAX_GAP = 8              # ★ これ以下の長さの欠損だけを埋める [サンプル]（frame_step=1 ならフレーム）
SMOOTH_WINDOW = 11       # ★ 平滑化の窓の幅 [サンプル]（奇数。大きいほど強くならす。30 fps なら 0.3〜0.5 秒ぶん）

clean = kodo.remove_outliers(tracks, threshold=OUTLIER_PX, window=7)
clean = kodo.interpolate_gaps(clean, max_gap=MAX_GAP)
clean = kodo.smooth_savgol(clean, window=SMOOTH_WINDOW, polyorder=2)
if MM_PER_PX is not None:
    clean = kodo.apply_scale(clean, MM_PER_PX)

for h in clean.history:
    print("・", h)
print(clean)

### 前処理の前と後を比べる

1 匹の最初の 10 秒を拡大して、処理の前（灰色の点）と後（線）を重ねます。飛んだ点が消え、線がなめらかになっていれば成功です。

In [ ]:
WHICH = 0                # ★ 比べる個体の番号（0, 1, 2, …）
n = min(int(10 * FPS / tracks.frame_step), tracks.n_frames)
raw = tracks.points(NODE)[WHICH, :n] * (MM_PER_PX or 1)
new = clean.points(NODE)[WHICH, :n]
fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(raw[:, 0], raw[:, 1], ".", color="0.6", ms=4, label="前処理の前")
ax.plot(new[:, 0], new[:, 1], "-", color="#2a78d6", lw=1.5, label="前処理の後")
ax.set_aspect("equal", adjustable="datalim"); ax.invert_yaxis()
ax.set_xlabel(f"x [{clean.units}]"); ax.set_ylabel(f"y [{clean.units}]")
ax.set_title(f"{clean.track_names[WHICH]}：最初の 10 秒"); ax.legend()
plt.show()

### ID の入れ替わりを確かめる

**何をする？** 2 匹がすれ違ったときに、追跡ソフトが名前（ID）を取り違えていないかを調べます。

**なぜ必要？** 入れ替わると、A の記録の後半が B のものになり、個体ごとの指標がまったく意味をなさなくなります。
疑いが出たら動画のそのフレームを見て、本当に入れ替わっていれば `kodo.swap_tracks(clean, "個体名1", "個体名2", from_sample=○)` で直します。

In [ ]:
swaps = kodo.detect_id_swaps(clean, node=NODE)
if len(swaps):
    display(swaps)
else:
    print("✓ ID の入れ替わりの疑いはありません")
# 例: clean = kodo.swap_tracks(clean, "track_0", "track_1", from_sample=1234)

## 7. 移動と静止を分けるしきい値を決める

**何をする？** 速さ（瞬間の速度）の分布を見て、「これより速ければ移動中」というしきい値を決めます。

**なぜ必要？** 静止していても、検出のぶれで速さは 0 になりません。分布の左側の山（静止中のぶれ）と右側の山（歩いているとき）の **谷** にしきい値を置くと、うまく分けられます。
山がはっきり分かれないときは、動画で「止まっている」と分かる区間の速さ（下の時系列のグラフで確認できます）より少し大きい値にします。

In [ ]:
v = kodo.speed(clean, node=NODE)
fig, ax = plt.subplots(figsize=(6.4, 3.6))
vv = v[np.isfinite(v)]
ax.hist(vv, bins=np.logspace(np.log10(max(vv[vv > 0].min(), 1e-3)), np.log10(vv.max()), 50),
        color="0.55", edgecolor="white")
ax.set_xscale("log")
ax.set_xlabel(f"速さ [{clean.units}/s]（対数目盛り）"); ax.set_ylabel("サンプル数")
ax.set_title("速さの分布（全個体）")
plt.show()

In [ ]:
SPEED_THRESHOLD = 3.0    # ★ 移動／静止のしきい値 [mm/s]（px のままなら px/s）
ax_v = kodo.plot_speed(clean, node=NODE, threshold=SPEED_THRESHOLD)
plt.show()

## 8. 調べたい領域（ROI）を決める

**何をする？** 容器の中に「えさ場」「中央」「すみ」などの領域を円や長方形で決めます。その領域にいた時間と、入った回数を数えます。

**なぜ必要？** 「明るい側と暗い側のどちらを好むか」「えさの近くにどれだけいたか」のような問いに答えるためです。

- 座標は **画像の px で決めて `.scaled(MM_PER_PX)` で mm に直す**のが簡単です（5. の画像で位置を読み取れます）。
- 円 `kodo.CircleROI(中心x, 中心y, 半径, "名前")`、長方形 `kodo.RectROI(左, 上, 右, 下, "名前")`。
- 練習用データの容器は画像の (40, 40)〜(640, 440) px です。

In [ ]:
ROIS_PX = [                                   # ★ 画像の px で領域を決める
    kodo.CircleROI(340, 240, 75, "中央"),
    kodo.RectROI(40, 40, 140, 140, "左上すみ"),
]
ROIS = [r.scaled(MM_PER_PX) for r in ROIS_PX] if MM_PER_PX else ROIS_PX

fig, ax = kodo.plot_trajectories(clean, node=NODE, rois=ROIS, title="軌跡と領域")
plt.show()

## 9. 指標を計算する

**何をする？** 個体ごとに指標を計算して表にします。

| 指標 | 定義 | 読み方 |
|---|---|---|
| 総移動距離 L | 隣り合うフレームの移動距離の合計 $\sum \lvert p_{t+1}-p_t\rvert$ | 活動量 |
| 平均の速さ | L ÷ 記録時間 | 静止時間も含めた平均 |
| 直進性 S | 始点と終点の直線距離 ÷ L | 1 = まっすぐ、0 に近いほどうろうろ |
| 移動の割合 | 速さ ≥ しきい値のフレームの割合 | 静止の割合 = 1 − 移動の割合 |
| 転向角 θ | 1 歩ごとの進む向きの変化（左 = ＋、右 = −） | 曲がりやすさ・左右の偏り |
| 交替性指数 | 曲がる向きが前回と逆だった割合 | 0.5 = でたらめ、1 に近い = 左右交互（ダンゴムシの交替性転向） |
| 領域の滞在時間・進入回数 | 領域内のフレーム数 × Δt、外→内の回数 | 好みの場所 |
| 最近接個体距離 | いちばん近い他個体までの距離の平均 | 集まる／避ける |
| MSD・拡散係数 D・指数 α | 時間 τ の間の移動距離の 2 乗の平均。$\text{MSD}=4D\tau$ | α≈1 ランダム、α≈2 直進、α<1 閉じこもり |

MSD と拡散係数 D は記録時間の 1 割までの τ で計算します。D は α が 0.7〜1.3（ランダムウォークとみなせる）のときだけ表に出ます。
領域に最初に入った時刻は、動画の 0 フレーム目を 0 秒として数えます。

`MIN_STEP` は転向角を計算するときの最小の移動距離です（必ず指定します）。高い fps では 1 フレームの移動がぶれと同じくらい小さく、角度がでたらめになるので、**体長の 1/4 くらい**進むごとに角度を測ります。

In [ ]:
MIN_STEP = 1.5           # ★ 転向角を測る最小の移動距離 [mm]（体長の 1/4 くらい）

summary = kodo.summarize(clean, node=NODE, speed_threshold=SPEED_THRESHOLD, rois=ROIS, min_step=MIN_STEP)
display(kodo.japanese_columns(summary).round(3).set_index("個体").T)

## 10. 図をかく

**何をする？** 4 種類の図（＋MSD）を描きます。白黒で印刷しても個体を区別できるよう、個体ごとに線の種類（実線・破線・点線）を変えています。
白黒印刷用に線を黒一色にしたいときは `kodo.set_print_mode(True)` を実行してから描きます。

### 軌跡
●が始点、■が終点です。y 軸は動画と同じく下向きです。

In [ ]:
fig, ax = kodo.plot_trajectories(clean, node=NODE, rois=ROIS)
plt.show()

### 転向角のヒストグラム
0° はまっすぐ、右側（＋）は左に曲がった、左側（−）は右に曲がったことを表します。山が 0° に鋭くとがっていれば直進的、左右に広ければよく曲がる個体です。

In [ ]:
fig, ax = kodo.plot_turning_hist(clean, node=NODE, min_step=MIN_STEP)
plt.show()

### 滞在ヒートマップ
容器を `BIN_SIZE` mm 四方の区画に分け、各区画にいた時間を濃さで表します（白 = 0 秒、黒 = 長い）。壁ぞいを好む個体は外周が濃くなります。

In [ ]:
BIN_SIZE = 5             # ★ 区画の一辺 [mm]
fig, axes = kodo.plot_heatmap(clean, bin_size=BIN_SIZE, node=NODE, rois=ROIS)
plt.show()

### 平均二乗変位（MSD）
両対数グラフで、点線（傾き 1）に平行ならランダムな動き、破線（傾き 2）に平行ならまっすぐ進む動きです。

In [ ]:
fig, ax = kodo.plot_msd(clean, node=NODE)
plt.show()

## 11. 結果を保存する

**何をする？** `results` フォルダに次のファイルを保存し、Colab ならまとめて zip でダウンロードします。

- `指標.csv` … 個体ごとの指標の表（Excel でそのまま開けます）
- `時系列.csv` … フレームごとの座標と速さ
- `軌跡.png` `速度.png` `転向角.png` `ヒートマップ.png` `MSD.png`
- `設定.txt` … fps・スケール・しきい値・前処理の記録

**なぜ必要？** 同じ設定でもう一度計算できるように（再現性）、**使った設定も一緒に残す**のが研究のきまりです。

In [ ]:
OUT = Path("results")
OUT.mkdir(exist_ok=True)
kodo.save_summary_csv(summary, OUT / "指標.csv")
kodo.save_timeseries_csv(clean, OUT / "時系列.csv", node=NODE)
fig_paths = kodo.save_all_figures(clean, OUT, node=NODE, speed_threshold=SPEED_THRESHOLD,
                                  min_step=MIN_STEP, bin_size=BIN_SIZE, rois=ROIS)
settings = [
    f"データ: {DATA_FILE}",
    f"fps: {FPS}",
    f"スケール: {MM_PER_PX} mm/px（定規 {RULER_P1}〜{RULER_P2} = {RULER_LENGTH_MM} mm）",
    f"部位: {NODE}",
    f"移動のしきい値: {SPEED_THRESHOLD} {clean.units}/s",
    f"転向角の最小移動距離: {MIN_STEP} {clean.units}",
    f"領域: {ROIS}",
    "前処理:",
] + ["  " + h for h in clean.history]
(OUT / "設定.txt").write_text("\n".join(settings), encoding="utf-8")

for p in sorted(OUT.iterdir()):
    print("保存:", p)

try:
    from google.colab import files
    import shutil
    shutil.make_archive("results", "zip", OUT)
    files.download("results.zip")
except ImportError:
    pass

## 12. ふりかえり：よくある失敗チェックリスト

- [ ] **スケール未設定**：表の単位が `px` のままになっていないか。カメラの高さが変わると比べられません。
- [ ] **フレームレートの誤り**：fps は撮影時の設定と同じか。スロー撮影の fps に注意。速度が「ありえない値」になっていたら最初に疑う。
- [ ] **ID の入れ替わり**：2 匹がすれ違った後、軌跡の色（線の種類）が入れかわっていないか。
- [ ] **見失いが多い**：`有効データの割合` が 0.8 未満の個体は、照明や背景を見直すか、その個体を除いて考える。
- [ ] **定規の高さ**：定規は動物が歩く面と同じ高さに置いて撮ったか（高さが違うと倍率が変わる）。
- [ ] **記録時間をそろえる**：直進性・移動距離・滞在時間は記録時間で変わる。比べる個体は同じ長さで。

練習が終わったら、`2.` のセルで `USE_DEMO = False` にして、自分のデータで上から実行し直しましょう。